In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


# FID and KID


## Config


In [ ]:
# per model/condition/species, against the held-out real set.
#
# KID is of primary importance. at n==100 FID is biased upward w9ith bias
# depending on the model; FID isn't comparable across conditions. KID is
# unbiased at any n. 
#

import os
import re
from pathlib import Path

ROOT = str(ROOT)
GEN     = os.path.join(ROOT, "outputs/generated")
BASE     = os.path.join(ROOT, "outputs/generated_baseline")
REF = os.path.join(ROOT, "data/reference")
TRAIN  = None          
METRICS       = os.path.join(ROOT, "outputs/metrics")
WORK      = "/content/metrics_work"

SPECIES = ["achillea", "carpobrotus", "eryngium"]
RES = 512
N_GEN = 100

os.makedirs(METRICS, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SKIP_FOLDERS = {"_smoketest"}

def list_images(folder):
    p = Path(folder)
    return sorted([f for f in p.rglob("*") if f.suffix.lower() in EXTS]) if p.exists() else []

def parse(folder_name, condition):
    name = folder_name.lower()
    if name in SKIP_FOLDERS:
        return None
    sp = next((s for s in SPECIES if s in name), None)
    if sp is None:
        return None
    model = name.split(sp)[0].strip("_") or "unnamed"
    return model, condition, sp

gen_files = {}
for root, cond in [(GEN, "lora"), (BASE, "base")]:
    if not os.path.exists(root):
        continue
    for folder in sorted(Path(root).iterdir()):
        if not folder.is_dir():
            continue
        parsed = parse(folder.name, cond)
        if parsed is None:
            print(f"  skipped: {folder.name}")
            continue
        gen_files[parsed] = list_images(folder)
        print(f"  {parsed[1]:<5} {parsed[0]:<8} {parsed[2]:<12} {len(gen_files[parsed]):>4}")

ref_files = {sp: list_images(Path(REF) / sp) for sp in SPECIES}
print()
for sp, f in ref_files.items():
    print(f"  reference {sp:<12} {len(f):>4}")


  lora  flux2    achillea      100
  lora  flux2    carpobrotus   100
  lora  flux2    eryngium      100
  lora  flux3    achillea      100
  lora  flux3    carpobrotus   100
  lora  flux3    eryngium      100
  lora  pixart   achillea      100
  lora  pixart   carpobrotus   100
  lora  pixart   eryngium      100
  lora  qwen     achillea      100
  lora  qwen     carpobrotus   100
  lora  qwen     eryngium      100
  lora  sdxl     achillea      100
  lora  sdxl     carpobrotus   100
  lora  sdxl     eryngium      100
  skipped: _smoketest
  base  flux2    achillea      100
  base  flux2    carpobrotus   100
  base  flux2    eryngium      100
  base  pixart   achillea      100
  base  pixart   carpobrotus   100
  base  pixart   eryngium      100
  base  qwen     achillea      100
  base  qwen     carpobrotus   100
  base  qwen     eryngium      100
  base  sdxl     achillea      100
  base  sdxl     carpobrotus   100
  base  sdxl     eryngium      100

  reference achillea      200
  

In [2]:
!pip install -q clean-fid torchmetrics

import torch
print("cuda:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.6 MB/s eta 0:00:00
CUDA: True


## Find the images


In [ ]:
# run first. if the folder layout has drifted from the
# config above, this is where it shows up.
from pathlib import Path

EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SKIP_FOLDERS = {"_smoketest"}

def list_images(folder):
    p = Path(folder)
    if not p.exists():
        return []
    return sorted([f for f in p.rglob("*") if f.suffix.lower() in EXTS])

def parse(folder_name, condition):
    name = folder_name.lower()
    if name in SKIP_FOLDERS:
        return None
    sp = next((s for s in SPECIES if s in name), None)
    if sp is None:
        return None
    model = name.split(sp)[0].strip("_") or "unnamed"
    return model, condition, sp

print("reference")
ref_files = {}
for sp in SPECIES:
    files = list_images(Path(REF) / sp)
    ref_files[sp] = files
    print(f"  {sp:<14} {len(files):>5} images")

print("\ngenerated")
gen_files = {}
for root, cond in [(GEN, "lora"), (BASE, "base")]:
    if not os.path.exists(root):
        print(f"  MISSING: {root}")
        continue
    for folder in sorted(Path(root).iterdir()):
        if not folder.is_dir():
            continue
        parsed = parse(folder.name, cond)
        if parsed is None:
            continue
        files = list_images(folder)
        gen_files[parsed] = files
        print(f"  {cond:<5} {parsed[0]:<8} {parsed[2]:<14} {len(files):>5}")

print(f"\n{len(gen_files)} cells found")


REFERENCE
  achillea         200 images
  carpobrotus      200 images
  eryngium         200 images

GENERATED
  lora  flux2    achillea         100
  lora  flux2    carpobrotus      100
  lora  flux2    eryngium         100
  lora  flux3    achillea         100
  lora  flux3    carpobrotus      100
  lora  flux3    eryngium         100
  lora  pixart   achillea         100
  lora  pixart   carpobrotus      100
  lora  pixart   eryngium         100
  lora  qwen     achillea         100
  lora  qwen     carpobrotus      100
  lora  qwen     eryngium         100
  lora  sdxl     achillea         100
  lora  sdxl     carpobrotus      100
  lora  sdxl     eryngium         100
  base  flux2    achillea         100
  base  flux2    carpobrotus      100
  base  flux2    eryngium         100
  base  pixart   achillea         100
  base  pixart   carpobrotus      100
  base  pixart   eryngium         100
  base  qwen     achillea         100
  base  qwen     carpobrotus      100
  base  qwen   

## Leakage check


In [ ]:
# a reference image that's also in training invalidates that condition.
# md5 catches exact dupes, not several photos of one plant from the same
# observation.
import hashlib

def md5_map(files):
    out = {}
    for f in files:
        out.setdefault(hashlib.md5(f.read_bytes()).hexdigest(), []).append(f)
    return out

if TRAIN and os.path.exists(TRAIN):
    train_hashes = set(md5_map(list_images(TRAIN)).keys())
    print(f"{len(train_hashes)} unique training images\n")

    for sp in SPECIES:
        ref_map = md5_map(ref_files[sp])
        overlap = set(ref_map) & train_hashes
        if overlap:
            print(f"  {sp}: {len(overlap)} LEAKED, removing")
            leaked = {f for h in overlap for f in ref_map[h]}
            ref_files[sp] = [f for f in ref_files[sp] if f not in leaked]
        else:
            print(f"  {sp}: clean")
        print(f"    {len(ref_files[sp])} reference images remain")
else:
    print("no TRAIN set, leakage unchecked")
    print("do not report these numbers until it is")


TRAINING_DIR not set or missing. Leakage NOT checked.
Do not report these numbers until it has been.


## Preprocess


In [ ]:
# both sets down the identical path so the metric cannot pick up a framing
# difference that exists only  in one of them. local disk, not Drive:
# clean  fid reads every file several times.
from PIL import Image
import shutil

def prepare(files, dest, limit=None):
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)

    written = 0
    for i, f in enumerate(files if limit is None else files[:limit]):
        try:
            im = Image.open(f).convert("RGB")
        except Exception as e:
            print(f"    skipping {f.name}: {e}")
            continue
        w, h = im.size
        side = min(w, h)
        im = im.crop(((w - side) // 2, (h - side) // 2,
                      (w + side) // 2, (h + side) // 2))
        im = im.resize((RES, RES), Image.BICUBIC)
        im.save(dest / f"{i:05d}.png")
        written += 1
    return written

print("reference")
ref_dirs = {}
for sp in SPECIES:
    d = Path(WORK) / "ref" / sp
    n = prepare(ref_files[sp], d)
    ref_dirs[sp] = str(d)
    print(f"  {sp:<14} {n:>5} images")

print("\ngenerated")
gen_dirs = {}
short = []
for k, files in gen_files.items():
    model, cond, sp = k
    d = Path(WORK) / "gen" / f"{model}_{cond}_{sp}"
    n = prepare(files, d, limit=N_GEN)
    gen_dirs[k] = str(d)
    if n < N_GEN:
        short.append((k, n))
    print(f"  {model:<8} {cond:<5} {sp:<14} {n:>5}")

if short:
    print("\nshort cells, FID is not comparable across sets of different size:")
    for k, n in short:
        print(f"  {k}: {n}")


Reference sets
  achillea         200 images
  carpobrotus      200 images
  eryngium         200 images

Generated sets
  flux2    lora  achillea         100
  flux2    lora  carpobrotus      100
  flux2    lora  eryngium         100
  flux3    lora  achillea         100
  flux3    lora  carpobrotus      100
  flux3    lora  eryngium         100
  pixart   lora  achillea         100
  pixart   lora  carpobrotus      100
  pixart   lora  eryngium         100
  qwen     lora  achillea         100
  qwen     lora  carpobrotus      100
  qwen     lora  eryngium         100
  sdxl     lora  achillea         100
  sdxl     lora  carpobrotus      100
  sdxl     lora  eryngium         100
  flux2    base  achillea         100
  flux2    base  carpobrotus      100
  flux2    base  eryngium         100
  pixart   base  achillea         100
  pixart   base  carpobrotus      100
  pixart   base  eryngium         100
  qwen     base  achillea         100
  qwen     base  carpobrotus      100
  qwe

## Compute


In [ ]:
# clean fid for FID as it keeps the resize step constant, which otherwise
# varies between libraries,  shifting FID on identical images.
from cleanfid import fid as cleanfid
from pathlib import Path
import pandas as pd

KID_SUBSET = 50

def n_images(d):
    return len(list(Path(d).glob("*.png")))

rows = []
for (model, cond, sp), gdir in sorted(gen_dirs.items()):
    rdir = ref_dirs[sp]
    print(f"{model} {cond} {sp} ...", end=" ", flush=True)

    f = cleanfid.compute_fid(rdir, gdir, mode="clean", num_workers=2)
    k = cleanfid.compute_kid(rdir, gdir, mode="clean", num_workers=2)

    rows.append({
        "model": model, "condition": cond, "species": sp,
        "n_real": n_images(rdir), "n_gen": n_images(gdir),
        "fid": round(f, 2),
        "kid_mean": round(k, 6),
    })
    print(f"FID {f:.1f}  KID {k:.5f}")

results = pd.DataFrame(rows)
results


flux2 base achillea ... compute FID between two folders
Found 200 images in the folder /content/metrics_work/ref/achillea


FID achillea : 100%|██████████| 7/7 [00:12<00:00,  1.75s/it]


Found 100 images in the folder /content/metrics_work/gen/flux2_base_achillea


FID flux2_base_achillea : 100%|██████████| 4/4 [00:06<00:00,  1.51s/it]


compute KID between two folders
Found 200 images in the folder /content/metrics_work/ref/achillea


...

KID eryngium : 100%|██████████| 7/7 [00:07<00:00,  1.02s/it]


Found 100 images in the folder /content/metrics_work/gen/sdxl_lora_eryngium


KID sdxl_lora_eryngium : 100%|██████████| 4/4 [00:06<00:00,  1.71s/it]


FID 194.1  KID 0.13706


,model,condition,species,n_real,n_gen,fid,kid_mean
0,flux2,base,achillea,200,100,203.64,0.132913
1,flux2,base,carpobrotus,200,100,194.78,0.139630
2,flux2,base,eryngium,200,100,227.24,0.177517
3,flux2,lora,achillea,200,100,186.03,0.112530
4,flux2,lora,carpobrotus,200,100,251.16,0.237906
5,flux2,lora,eryngium,200,100,260.57,0.252433
6,flux3,lora,achillea,200,100,158.90,0.079588
7,flux3,lora,carpobrotus,200,100,224.01,0.193634
8,flux3,lora,eryngium,200,100,206.20,0.162900
9,pixart,base,achillea,200,100,195.69,0.110618


## Aggregate


In [ ]:
# resample the generated set and recompute KID, to see whether a gap
# between two conditions is larger than the noise. overlapping intervals
# mean this metric doesn't separate them-
by_arm = (
    results.groupby(["model", "condition"])
    .agg(fid=("fid", "mean"), kid=("kid_mean", "mean"),
         kid_sd=("kid_mean", "std"), n=("kid_mean", "size"))
    .round(4)
    .sort_values("kid")
)
by_arm


fid     kid  kid_sd  n
model  condition                             
pixart lora       164.7900  0.0815  0.0185  3
sdxl   lora       175.7133  0.0991  0.0331  3
flux3  lora       196.3700  0.1454  0.0590  3
flux2  base       208.5533  0.1500  0.0240  3
sdxl   base       213.9500  0.1572  0.0607  3
pixart base       220.2600  0.1573  0.0880  3
qwen   lora       220.7100  0.1790  0.0603  3
       base       253.1967  0.1982  0.0862  3
flux2  lora       232.5867  0.2010  0.0769  3

In [ ]:
# --- prepare the training sets (run once) ------------------------------
TRAIN = os.path.join(ROOT, "data/train")

# species key -> the folder that contains LoRA species-specific training images.
TRAIN_FOLDERS = {
    "achillea":    ["Achillea_Maritima_2"],
    "carpobrotus": ["Carpobrotus_Acinaciformis_3"],
    "eryngium":    ["Eryngium_Maritimum_2"],
}

train_dirs = {}
for sp, folders in TRAIN_FOLDERS.items():
    files = []
    for fo in folders:
        files += list_images(Path(TRAIN) / fo)
    d = Path(WORK) / "train" / sp
    n = prepare(files, d)
    train_dirs[sp] = str(d)
    print(f"  {sp:<12} {n:>4} training images")


  achillea      138 training images
  carpobrotus    70 training images
  eryngium       90 training images


In [29]:
rows = []
for (model, cond, sp), gdir in sorted(gen_dirs.items()):
    print(f"{model} {cond} {sp} ...", end=" ", flush=True)
    row = {"model": model, "condition": cond, "species": sp,
           "n_gen": n_images(gdir)}

    for label, ref in [("heldout", ref_dirs[sp]), ("train", train_dirs[sp])]:
        row[f"fid_{label}"] = round(
            cleanfid.compute_fid(ref, gdir, mode="clean", num_workers=2), 2)
        row[f"kid_{label}"] = round(
            cleanfid.compute_kid(ref, gdir, mode="clean", num_workers=2), 6)
        row[f"n_{label}"] = n_images(ref)

    rows.append(row)
    print(f"KID heldout {row['kid_heldout']:.4f}  train {row['kid_train']:.4f}")

results = pd.DataFrame(rows)
results


flux2 base achillea ... compute FID between two folders
Found 200 images in the folder /content/metrics_work/ref/achillea


FID achillea : 100%|██████████| 7/7 [00:07<00:00,  1.05s/it]


Found 100 images in the folder /content/metrics_work/gen/flux2_base_achillea


FID flux2_base_achillea : 100%|██████████| 4/4 [00:06<00:00,  1.60s/it]


compute KID between two folders
Found 200 images in the folder /content/metrics_work/ref/achillea


...

KID eryngium : 100%|██████████| 3/3 [00:05<00:00,  1.95s/it]


Found 100 images in the folder /content/metrics_work/gen/sdxl_lora_eryngium


KID sdxl_lora_eryngium : 100%|██████████| 4/4 [00:07<00:00,  1.77s/it]


KID heldout 0.1368  train 0.0604


,model,condition,species,n_gen,fid_heldout,kid_heldout,n_heldout,fid_train,kid_train,n_train
0,flux2,base,achillea,100,203.64,0.133553,200,259.34,0.201695,138
1,flux2,base,carpobrotus,100,194.78,0.139550,200,252.51,0.188706,70
2,flux2,base,eryngium,100,227.24,0.177138,200,308.25,0.275713,90
3,flux2,lora,achillea,100,186.03,0.111713,200,140.83,0.056069,138
4,flux2,lora,carpobrotus,100,251.16,0.239222,200,163.65,0.071239,70
5,flux2,lora,eryngium,100,260.57,0.253881,200,193.08,0.141528,90
6,flux3,lora,achillea,100,158.90,0.078567,200,116.35,0.031428,138
7,flux3,lora,carpobrotus,100,224.01,0.192275,200,141.55,0.036349,70
8,flux3,lora,eryngium,100,206.20,0.162661,200,125.33,0.047588,90
9,pixart,base,achillea,100,195.69,0.111342,200,254.87,0.173771,138


In [30]:
print("real vs real, the floor at these sample sizes:")
for sp in SPECIES:
    k = cleanfid.compute_kid(train_dirs[sp], ref_dirs[sp], mode="clean", num_workers=2)
    f = cleanfid.compute_fid(train_dirs[sp], ref_dirs[sp], mode="clean", num_workers=2)
    print(f"  {sp:<12} FID {f:6.1f}   KID {k:.5f}   "
          f"(n={n_images(train_dirs[sp])} vs {n_images(ref_dirs[sp])})")


Real vs real (training vs held-out), the floor for these sample sizes:
compute KID between two folders
Found 138 images in the folder /content/metrics_work/train/achillea


KID achillea : 100%|██████████| 5/5 [00:06<00:00,  1.34s/it]


Found 200 images in the folder /content/metrics_work/ref/achillea


KID achillea : 100%|██████████| 7/7 [00:08<00:00,  1.27s/it]


compute FID between two folders
Found 138 images in the folder /content/metrics_work/train/achillea


...

compute FID between two folders
Found 90 images in the folder /content/metrics_work/train/eryngium


FID eryngium : 100%|██████████| 3/3 [00:05<00:00,  1.99s/it]


Found 200 images in the folder /content/metrics_work/ref/eryngium


FID eryngium : 100%|██████████| 7/7 [00:08<00:00,  1.25s/it]


  eryngium     FID  143.0   KID 0.05280   (n=90 vs 200)


In [27]:
import numpy as np, shutil, os
from pathlib import Path
from cleanfid import fid as cleanfid
import pandas as pd

N_BOOT = 10
rng = np.random.default_rng(42)
BOOT_DIR = Path("/content/boot_tmp")

boot_rows = []
for (model, cond, sp), gdir in sorted(gen_dirs.items()):
    files = sorted(Path(gdir).glob("*.png"))
    vals = []
    for b in range(N_BOOT):
        if BOOT_DIR.exists():
            shutil.rmtree(BOOT_DIR)
        BOOT_DIR.mkdir(parents=True)
        idx = rng.choice(len(files), size=len(files), replace=True)
        for j, i in enumerate(idx):
            os.link(files[i], BOOT_DIR / f"{j:05d}.png")
        vals.append(cleanfid.compute_kid(ref_dirs[sp], str(BOOT_DIR),
                                         mode="clean", num_workers=2))

    lo, hi = np.percentile(vals, [2.5, 97.5])
    boot_rows.append({
        "model": model, "condition": cond, "species": sp,
        "kid_mean": round(float(np.mean(vals)), 6),
        "ci_lo": round(float(lo), 6), "ci_hi": round(float(hi), 6),
    })
    print(f"{model} {cond} {sp}: {np.mean(vals):.5f} [{lo:.5f}, {hi:.5f}]")

bootstrap = pd.DataFrame(boot_rows)
bootstrap


compute KID between two folders
Found 200 images in the folder /content/metrics_work/ref/achillea


KID achillea : 100%|██████████| 7/7 [00:10<00:00,  1.55s/it]


Found 100 images in the folder /content/boot_tmp


KID boot_tmp : 100%|██████████| 4/4 [00:05<00:00,  1.32s/it]


compute KID between two folders
Found 200 images in the folder /content/metrics_work/ref/achillea


...

Found 100 images in the folder /content/boot_tmp


KID boot_tmp : 100%|██████████| 4/4 [00:06<00:00,  1.65s/it]


compute KID between two folders
Found 200 images in the folder /content/metrics_work/ref/carpobrotus


KID carpobrotus :   0%|          | 0/7 [00:01<?, ?it/s]


KeyboardInterrupt: 

## Export


In [ ]:
results.to_csv(os.path.join(METRICS, "distributional_metrics.csv"), index=False)
by_arm.to_csv(os.path.join(METRICS, "distributional_by_arm.csv"))
bootstrap.to_csv(os.path.join(METRICS, "distributional_bootstrap.csv"), index=False)

print("wrote to", METRICS)
for f in ["distributional_metrics.csv", "distributional_by_arm.csv",
          "distributional_bootstrap.csv"]:
    print(" ", f)


In [ ]:
# LaTeX table. KID is scaled by 1000 for readability; state this in the caption.
tex = results.pivot_table(index=["model", "condition"],
                          columns="species", values="kid_mean")

lines = [
    r"\begin{table}[htbp]", r"\centering", r"\footnotesize",
    r"\setlength{\tabcolsep}{4pt}",
    r"\caption{Kernel Inception Distance ($\times 10^{3}$, lower is better) "
    r"against held-out reference images.}",
    r"\label{tab:kid}",
    r"\begin{tabular}{ll" + "c" * len(SPECIES) + "}", r"\toprule",
    "Model & Cond. & " + " & ".join(s.title() for s in tex.columns) + r" \\",
    r"\midrule",
]
for (model, cond), row in tex.iterrows():
    cells = " & ".join(f"{row[s] * 1000:.1f}" for s in tex.columns)
    lines.append(f"{model} & {cond} & {cells} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

table_tex = "\n".join(lines)
with open(os.path.join(METRICS, "kid_table.tex"), "w") as f:
    f.write(table_tex)
print(table_tex)
